In [1]:
import os
#os.environ["KERAS_BACKEND"] = "jax"  # can use tensorflow
os.environ["KERAS_BACKEND"] = "tensorflow"  # Or "jax" or "torch"!

import json
import tensorflow as tf
import numpy as np
import pandas as pd
import plotly.graph_objs as go
import plotly.express as px
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split



import keras
from keras import ops
import keras_nlp


#import deberta_v3_small_en
#from deberta_v3_small_en import DebertaV3Tokenizer

# Initialize tokenizer
#tokenizer = DebertaV3Tokenizer.from_pretrained('C:\\Users\\帅的一\\.cache\\kagglehub\\models\\keras\\deberta_v3\\keras\\deberta_v3_small_en\\2')


# Define CFG class here

class CFG:
    seed = 42
    preset = "deberta_v3_small_en" # name of pretrained backbone 'deberta_v3_small_en'
    train_seq_len = 1024 # max size of input sequence for training
    train_batch_size = 2 * 8 # size of the input batch in training, x 2 as two GPUs
    infer_seq_len = 2000 # max size of input sequence for inference
    infer_batch_size = 2 * 2 # size of the input batch in inference, x 2 as two GPUs
    epochs = 6 # number of epochs to train
    lr_mode = "exp" # lr scheduler mode from one of "cos", "step", "exp"
    
    labels = ["B-EMAIL", "B-ID_NUM", "B-NAME_STUDENT", "B-PHONE_NUM",
              "B-STREET_ADDRESS", "B-URL_PERSONAL", "B-USERNAME",
              "I-ID_NUM", "I-NAME_STUDENT", "I-PHONE_NUM",
              "I-STREET_ADDRESS","I-URL_PERSONAL","O"]
    id2label = dict(enumerate(labels)) # integer label to BIO format label mapping
    label2id = {v:k for k,v in id2label.items()} # BIO format label to integer label mapping
    num_labels = len(labels) # number of PII (NER) tags
    
    train = True # whether to train or use already trained ckpt
    
    keras.utils.set_random_seed(seed)
    
    # Get devices default "gpu" or "tpu"
devices = keras.distribution.list_devices()
print("Device:", devices)

# Rest of my code follows.




import plotly.graph_objs as go
import plotly.express as px
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split

import tensorflow as tf
import keras
import keras_nlp
from keras import ops


import keras
import keras_nlp
from keras import ops
import tensorflow as tf

import json
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split

import plotly.graph_objs as go
import plotly.express as px

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
#print("KerasNLP:", keras_nlp.__version__)

class CFG:
    seed = 42
    preset = "deberta_v3_small_en" # name of pretrained backbone
#    model = keras_nlp.models
    train_seq_len = 1024 # max size of input sequence for training
    train_batch_size = 2 * 8 # size of the input batch in training, x 2 as two GPUs
    infer_seq_len = 2000 # max size of input sequence for inference
    infer_batch_size = 2 * 2 # size of the input batch in inference, x 2 as two GPUs
    epochs = 6 # number of epochs to train
    lr_mode = "exp" # lr scheduler mode from one of "cos", "step", "exp"
    #B- indicates the beginning of an PII, I- indicates an inner part of a multi-token PII, 
    #and O indicates tokens that do not belong to any PII.
    labels = ["B-EMAIL", "B-ID_NUM", "B-NAME_STUDENT", "B-PHONE_NUM",
              "B-STREET_ADDRESS", "B-URL_PERSONAL", "B-USERNAME",
              "I-ID_NUM", "I-NAME_STUDENT", "I-PHONE_NUM",
              "I-STREET_ADDRESS","I-URL_PERSONAL","O"]
    id2label = dict(enumerate(labels)) # integer label to BIO format label mapping
    label2id = {v:k for k,v in id2label.items()} # BIO format label to integer label mapping
    num_labels = len(labels) # number of PII (NER) tags
    
    train = True # whether to train or use already trained ckpt
    
    keras.utils.set_random_seed(CFG.seed)
    
    # Get devices default "gpu" or "tpu"
devices = keras.distribution.list_devices()
print("Device:", devices)

if len(devices) > 1:
    # Data parallelism
    data_parallel = keras.distribution.MultiWorkerMirroredStrategy(devices=devices)

    # Set the global distribution.
    keras.distribution.set_distribution_strategy(data_parallel)

keras.mixed_precision.set_global_policy("mixed_float16")

BASE_PATH = "/kaggle/input/pii-detection-removal-from-educational-data"

# Train-Valid data
data = json.load(open(f"{BASE_PATH}/train.json"))

# Initialize empty arrays
words = np.empty(len(data), dtype=object)
labels = np.empty(len(data), dtype=object)

# Fill the arrays
for i, x in tqdm(enumerate(data), total=len(data)):
    words[i] = np.array(x["tokens"])
    labels[i] = np.array([CFG.label2id[label] for label in x["labels"]])

# Get unique labels and their frequency
all_labels = np.array([x for label in labels for x in label])
unique_labels, label_counts = np.unique(all_labels, return_counts=True)

# Plotting
fig = go.Figure(data=go.Bar(x=CFG.labels, y=label_counts))
fig.update_layout(
    title="Label Distribution",
    xaxis_title="Labels",
    yaxis_title="Count",
    yaxis_type="log",
)

fig.update_traces(text=label_counts, textposition="outside")
fig.show()

# Splitting the data into training and testing sets
train_words, valid_words, train_labels, valid_labels = train_test_split(
    words, labels, test_size=0.2, random_state=CFG.seed
)

# To convert string input or list of strings input to numerical tokens
tokenizer = keras_nlp.models.DebertaV3Tokenizer.from_preset(
    CFG.preset,
)

# Preprocessing layer to add special tokens: [CLS], [SEP], [PAD]
packer = keras_nlp.layers.MultiSegmentPacker(
    start_value=tokenizer.cls_token_id,
    end_value=tokenizer.sep_token_id,
    sequence_length=10,
)

#sample_words = words[0][:5]
#sample_tokens_int = [
#    token.tolist() for word in sample_words for token in tokenizer(word)
#]
#sample_tokens_str = [tokenizer.id_to_token(token) for token in sample_tokens_int]

#print("words        :", sample_words.tolist())
#print("tokens (str) :", sample_tokens_str)
#print("tokens (int) :", sample_tokens_int)

# Define sample words
sample_words = words[0][:5]  # Assuming 'words' is correctly defined

# Tokenize sample words
sample_tokens_int = []
tf.experimental.numpy.experimental_enable_numpy_behavior()
for word in sample_words:
    tokens = tokenizer(word)  # Assuming tokenizer correctly tokenizes words
    #sample_tokens_int.extend(tokens.tolist())  # Assuming tokenizer output is a tensor
    sample_tokens_int.extend(tokens.tolist())  # Corrected line

# Convert token IDs to strings using tokenizer
sample_tokens_str = [tokenizer.id_to_token(token) for token in sample_tokens_int]

# Print outputs for debugging
print("Sample Words  :", sample_words.tolist())
print("Token IDs     :", sample_tokens_int)
print("Token Strings :", sample_tokens_str)


padded_sample_tokens_int = packer(np.array(sample_tokens_int))[0].tolist()
padded_sample_tokens_str = [
    tokenizer.id_to_token(token) for token in padded_sample_tokens_int
]

print("tokens (str)        :", sample_tokens_str)
print("padded tokens (str) :", padded_sample_tokens_str, "\n")

print("tokens (int)        :", sample_tokens_int)
print("padded tokens (int) :", padded_sample_tokens_int)


def get_tokens(words, seq_len, packer):
    # Tokenize input
    token_words = tf.expand_dims(
        tokenizer(words), axis=-1
    )  # ex: (words) ["It's", "a", "cat"] ->  (token_words) [[1, 2], [3], [4]]
    tokens = tf.reshape(
        token_words, [-1]
    )  # ex: (token_words) [[1, 2], [3], [4]] -> (tokens) [1, 2, 3, 4]
    # Pad tokens
    tokens = packer(tokens)[0][:seq_len]
    inputs = {"token_ids": tokens, "padding_mask": tokens != 0}
    return inputs, tokens, token_words


def get_token_ids(token_words):
    # Get word indices
    word_ids = tf.range(tf.shape(token_words)[0])
    # Get size of each word
    word_size = tf.reshape(tf.map_fn(lambda word: tf.shape(word)[0:1], token_words), [-1])
    # Repeat word_id with size of word to get token_id
    token_ids = tf.repeat(word_ids, word_size)
    return token_ids


def get_token_labels(word_labels, token_ids, seq_len):
    # Create token_labels from word_labels ->  alignment
    token_labels = tf.gather(word_labels, token_ids)
    # Only label the first token of a given word and assign -100 to others
    mask = tf.concat([[True], token_ids[1:] != token_ids[:-1]], axis=0)
    token_labels = tf.where(mask, token_labels, -100)
    # Truncate to max sequence length
    token_labels = token_labels[: seq_len - 2]  # -2 for special tokens ([CLS], [SEP])
    # Pad token_labels to align with tokens (use -100 to pad for loss/metric ignore)
    pad_start = 1  # for [CLS] token
    pad_end = seq_len - tf.shape(token_labels)[0] - 1  # for [SEP] and [PAD] tokens
    token_labels = tf.pad(token_labels, [[pad_start, pad_end]], constant_values=-100)
    return token_labels


def process_token_ids(token_ids, seq_len):
    # Truncate to max sequence length
    token_ids = token_ids[: seq_len - 2]  # -2 for special tokens ([CLS], [SEP])
    # Pad token_ids to align with tokens (use -1 to pad for later identification)
    pad_start = 1  # [CLS] token
    pad_end = seq_len - tf.shape(token_ids)[0] - 1  # [SEP] and [PAD] tokens
    token_ids = tf.pad(token_ids, [[pad_start, pad_end]], constant_values=-1)
    return token_ids


def process_data(seq_len=720, has_label=True, return_ids=False):
    # To add special tokens: [CLS], [SEP], [PAD]
    packer = keras_nlp.layers.MultiSegmentPacker(
        start_value=tokenizer.cls_token_id,
        end_value=tokenizer.sep_token_id,
        sequence_length=seq_len,
    )

    def process(x):
        # Generate inputs from tokens
        inputs, tokens, words_int = get_tokens(x["words"], seq_len, packer)
        # Generate token_ids for mapping tokens to words
        token_ids = get_token_ids(words_int)
        if has_label:
            # Generate token_labels from word_labels
            token_labels = get_token_labels(x["labels"], token_ids, seq_len)
            return inputs, token_labels
        elif return_ids:
            # Pad token_ids to align with tokens
            token_ids = process_token_ids(token_ids, seq_len)
            return token_ids
        else:
            return inputs

    return process


def build_dataset(words, labels=None, return_ids=False, batch_size=4,
                  seq_len=512, shuffle=False, cache=True, drop_remainder=True):
    AUTO = tf.data.AUTOTUNE

    slices = {"words": tf.ragged.constant(words)}
    if labels is not None:
        slices.update({"labels": tf.ragged.constant(labels)})

    ds = tf.data.Dataset.from_tensor_slices(slices)
    ds = ds.map(process_data(seq_len=seq_len,
                             has_label=labels is not None,
                             return_ids=return_ids), num_parallel_calls=AUTO)  # apply processing
    ds = ds.cache() if cache else ds  # cache dataset
    if shuffle:  # shuffle dataset
        ds = ds.shuffle(1024, seed=CFG.seed)
        opt = tf.data.Options()
        opt.experimental_deterministic = False
        ds = ds.with_options(opt)
    ds = ds.batch(batch_size, drop_remainder=drop_remainder)  # batch dataset
    ds = ds.prefetch(AUTO)  # prefetch next batch
    return ds


train_ds = build_dataset(train_words, train_labels, batch_size=CFG.train_batch_size,
                         seq_len=CFG.train_seq_len, shuffle=True)

valid_ds = build_dataset(valid_words, valid_labels, batch_size=CFG.train_batch_size,
                         seq_len=CFG.train_seq_len, shuffle=False)

inp, tar = next(iter(valid_ds))
print("# Input:\n", inp)
print("\n# Labels:\n", tar)

class CrossEntropy(keras.losses.SparseCategoricalCrossentropy):
    def __init__(self, ignore_class=-100, reduction=None, **args):
        super().__init__(reduction=reduction, **args)
        self.ignore_class = ignore_class

    def call(self, y_true, y_pred):
        y_true = ops.reshape(y_true, [-1])
        y_pred = ops.reshape(y_pred, [-1, CFG.num_labels])
        loss = super().call(y_true, y_pred)
        if self.ignore_class is not None:
            valid_mask = ops.not_equal(
                y_true, ops.cast(self.ignore_class, y_pred.dtype)
            )
            loss = ops.where(valid_mask, loss, 0.0)
            loss = ops.sum(loss)
            loss /= ops.maximum(ops.sum(ops.cast(valid_mask, loss.dtype)), 1)
        else:
            loss = ops.mean(loss)
        return loss


import tensorflow.keras.backend as K

class FBetaScore(tf.keras.metrics.Metric):
    def __init__(self, ignore_classes=[-100, 12], average="micro", beta=5.0, name="f5_score", **kwargs):
        super(FBetaScore, self).__init__(name=name, **kwargs)
        self.beta = beta
        self.average = average
        self.ignore_classes = ignore_classes
        self.true_positives = self.add_weight(name="tp", initializer="zeros")
        self.false_positives = self.add_weight(name="fp", initializer="zeros")
        self.false_negatives = self.add_weight(name="fn", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.cast(tf.argmax(y_pred, axis=-1), tf.int32)
        
        mask = tf.ones_like(y_true, dtype=tf.bool)
        for ignore_class in self.ignore_classes:
            mask &= tf.not_equal(y_true, ignore_class)
        
        y_true = tf.boolean_mask(y_true, mask)
        y_pred = tf.boolean_mask(y_pred, mask)
        
        tp = tf.cast(tf.math.count_nonzero(y_pred * y_true), tf.float32)
        fp = tf.cast(tf.math.count_nonzero(y_pred * (1 - y_true)), tf.float32)
        fn = tf.cast(tf.math.count_nonzero((1 - y_pred) * y_true), tf.float32)
        
        self.true_positives.assign_add(tp)
        self.false_positives.assign_add(fp)
        self.false_negatives.assign_add(fn)

    def result(self):
        precision = self.true_positives / (self.true_positives + self.false_positives + K.epsilon())
        recall = self.true_positives / (self.true_positives + self.false_negatives + K.epsilon())

        if self.average == "micro":
            precision = K.mean(precision)
            recall = K.mean(recall)

        fbeta = (1 + self.beta ** 2) * (precision * recall) / ((self.beta ** 2) * precision + recall + K.epsilon())
        return fbeta

    def reset_states(self):
        self.true_positives.assign(0)
        self.false_positives.assign(0)
        self.false_negatives.assign(0)


# Build Token Classification model
backbone = keras_nlp.models.DebertaV3Backbone.from_preset(
    CFG.preset,
    seq_len=CFG.train_seq_len  # Add seq_len parameter
)
out = backbone.output
out = keras.layers.Dense(CFG.num_labels, name="logits")(out)
out = keras.layers.Activation("softmax", dtype="float32", name="prediction")(out)
model = keras.models.Model(backbone.input, out)

# Compile model for optimizer, loss, and metric
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=2e-5),
    loss=CrossEntropy(),
    metrics=[FBetaScore()],
)

# Summary of the model architecture
model.summary()

import math


def get_lr_callback(batch_size=8, mode='cos', epochs=10, plot=False):
    lr_start, lr_max, lr_min = 6e-6, 2.5e-6 * batch_size, 1e-6
    lr_ramp_ep, lr_sus_ep, lr_decay = 3, 0, 0.75

    def lrfn(epoch):  # Learning rate update function
        if mode == 'exp':
            lr = lr_max * lr_decay ** epoch
        elif mode == 'cos':
            if epoch < lr_ramp_ep:
                lr = (lr_max - lr_start) / lr_ramp_ep * epoch + lr_start
            elif epoch < lr_ramp_ep + lr_sus_ep:
                lr = lr_max
            else:
                lr = (lr_max - lr_min) * 0.5 * (
                            math.cos((epoch - lr_ramp_ep - lr_sus_ep) / (epochs - lr_ramp_ep - lr_sus_ep) * math.pi) + 1) + lr_min
        elif mode == 'step':
            if epoch < 2:
                lr = 1e-5
            elif epoch < 4:
                lr = 5e-6
            else:
                lr = 1e-6
        else:
            lr = lr_max

        return lr

    if plot:
        plt.figure(figsize=(10, 5))
        plt.plot(np.arange(epochs), [lrfn(x) for x in range(epochs)])
        plt.xlabel('Epoch')
        plt.ylabel('Learning Rate')
        plt.title(f'{mode} Learning Rate Schedule')
        plt.show()

    lr_callback = tf.keras.callbacks.LearningRateScheduler(lrfn, verbose=False)

    return lr_callback


callbacks = [
    get_lr_callback(batch_size=CFG.train_batch_size,
                    mode=CFG.lr_mode, epochs=CFG.epochs),
]

if not os.path.exists("logs"):
    os.mkdir("logs")

# Train model
history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=CFG.epochs,
    callbacks=callbacks,
    verbose=1,
)



2024-05-04 00:27:45.004840: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-05-04 00:27:45.004987: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-05-04 00:27:45.127162: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Device: ['cpu:0']
TensorFlow: 2.15.0
Keras: 3.1.1
Device: ['cpu:0']


  0%|          | 0/6807 [00:00<?, ?it/s]

Attaching 'tokenizer.json' from model 'keras/deberta_v3/keras/deberta_v3_small_en/2' to your Kaggle notebook...
Attaching 'tokenizer.json' from model 'keras/deberta_v3/keras/deberta_v3_small_en/2' to your Kaggle notebook...
Attaching 'assets/tokenizer/vocabulary.spm' from model 'keras/deberta_v3/keras/deberta_v3_small_en/2' to your Kaggle notebook...


Sample Words  : ['Design', 'Thinking', 'for', 'innovation', 'reflexion']
Token IDs     : [2169, 12103, 270, 3513, 28310, 4593]
Token Strings : ['▁Design', '▁Thinking', '▁for', '▁innovation', '▁reflex', 'ion']
tokens (str)        : ['▁Design', '▁Thinking', '▁for', '▁innovation', '▁reflex', 'ion']
padded tokens (str) : ['[CLS]', '▁Design', '▁Thinking', '▁for', '▁innovation', '▁reflex', 'ion', '[SEP]', '[PAD]', '[PAD]'] 

tokens (int)        : [2169, 12103, 270, 3513, 28310, 4593]
padded tokens (int) : [1, 2169, 12103, 270, 3513, 28310, 4593, 2, 0, 0]


Attaching 'config.json' from model 'keras/deberta_v3/keras/deberta_v3_small_en/2' to your Kaggle notebook...


# Input:
 {'token_ids': <tf.Tensor: shape=(16, 1024), dtype=int32, numpy=
array([[    1, 28525,   877, ...,     0,     0,     0],
       [    1, 45730,   377, ...,     0,     0,     0],
       [    1,  8489,  7933, ...,     0,     0,     0],
       ...,
       [    1,  6348, 28525, ...,  1068,  3955,     2],
       [    1,  3780, 13942, ...,     0,     0,     0],
       [    1, 45730,   377, ...,  6568,   366,     2]], dtype=int32)>, 'padding_mask': <tf.Tensor: shape=(16, 1024), dtype=bool, numpy=
array([[ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ..., False, False, False],
       ...,
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ...,  True,  True,  True]])>}

# Labels:
 tf.Tensor(
[[-100   12   12 ... -100 -100 -100]
 [-100   12   12 ... -100 -100 -100]
 [-100   12 -100 ... -100 -100 -100]
 ...
 [-100 

Attaching 'config.json' from model 'keras/deberta_v3/keras/deberta_v3_small_en/2' to your Kaggle notebook...


TypeError: <class 'keras_nlp.src.models.deberta_v3.deberta_v3_backbone.DebertaV3Backbone'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras_nlp.src.models.deberta_v3.deberta_v3_backbone', 'class_name': 'DebertaV3Backbone', 'config': {'name': 'deberta_v3_backbone', 'trainable': True, 'vocabulary_size': 128100, 'num_layers': 6, 'num_heads': 12, 'hidden_dim': 768, 'intermediate_dim': 3072, 'dropout': 0.1, 'max_sequence_length': 512, 'bucket_size': 256, 'seq_len': 1024}, 'registered_name': 'keras_nlp>DebertaV3Backbone', 'assets': [], 'weights': 'model.weights.h5'}.

Exception encountered: Function.__init__() got an unexpected keyword argument 'seq_len'